# Ch10. Dynamic Regression
**Forecasting: Principles & Practice (Python Edition)**  
Lab Notebook (74 slides) · [github.com/bcseong2/fpppy-labs](https://github.com/bcseong2/fpppy-labs)

In [ ]:
%pip install statsforecast neuralforecast hierarchicalforecast mlforecast utilsforecast

## [Slide 8] 10.2 StatsForecast Implementation

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, ARIMA

# Automatic ARIMA order selection for errors
sf = StatsForecast(
    models=[AutoARIMA(season_length=1)],
    freq='QS'
)
sf.fit(df=df, target_col="y")  # other columns = predictors

## [Slide 13-15] 10.2 US Consumption: Code

In [ ]:
import pandas as pd
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA
from statsforecast.arima import ARIMASummary

us_change = (
    pd.read_csv("data/US_change.csv", parse_dates=["ds"])
    .rename(columns={"y": "Consumption"})
    [["unique_id", "ds", "Consumption", "Income"]]
)

sf = StatsForecast(models=[AutoARIMA()], freq="QS")
sf = sf.fit(df=us_change, target_col="Consumption")

model = sf.fitted_[0, 0].model_
print(ARIMASummary(model))
print(pd.Series(model["coef"]).round(3))
print({k: round(float(model[k]), 2)
       for k in ["sigma2", "loglik", "aicc"]})

## [Slide 16] 10.2 US Consumption: Residuals — Code

In [ ]:
import matplotlib.pyplot as plt
coef = model["coef"]
eta_hat = (us_change["Consumption"] - coef["intercept"]
           - coef["ex_1"] * us_change["Income"])
resid = pd.DataFrame({
    "ds": us_change["ds"],
    "Regression residuals": eta_hat,        # eta_hat
    "ARIMA residuals": model["residuals"],  # epsilon_hat
})

fig, axes = plt.subplots(2, 1, sharex=True, figsize=(7, 3.6))
for ax, col in zip(axes, ["Regression residuals",
                          "ARIMA residuals"]):
    ax.plot(resid["ds"], resid[col], color="black", lw=0.9)
    ax.set_title(col, fontsize=9)
plt.tight_layout(); plt.show()

## [Slide 19] 10.2 US Consumption: Ljung-Box Test

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox
acorr_ljungbox(model["residuals"], lags=[8], model_df=3)
#    lb_stat  lb_pvalue
# 8    5.207      0.391

## [Slide 21] 10.3 Forecasting: US Consumption

In [ ]:
from utilsforecast.processing import make_future_dataframe

# Assumption: future income changes = historical mean
future_predictors = (
    make_future_dataframe(uids=sf.uids,
                          last_times=sf.last_dates,
                          freq="QS", h=8)
    .assign(Income=us_change["Income"].mean())
)

fc = sf.forecast(
    df=us_change, h=8,
    X_df=future_predictors,
    target_col="Consumption",
    level=[80, 95]
)
fc.round(2)

## [Slide 26-27, 29, 34] 10.3 Electricity Demand: Data, Code and Forecast

In [ ]:
import pandas as pd
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA
from statsforecast.arima import ARIMASummary
from statsmodels.stats.diagnostic import acorr_ljungbox

vic = pd.read_csv("data/vic_elec.csv", parse_dates=["ds"])
vic = vic[vic["ds"] >= "2014"]   # half-hourly, long format
wide = vic.pivot(index="ds", columns="unique_id", values="y")
hol = (vic[vic["unique_id"] == "Demand"]
       .set_index("ds")["Holiday"])
daily = wide.resample("D").agg({"Demand": "sum",
                                "Temperature": "max"})
daily["Demand"] /= 1e3           # MW -> GW (daily total)
daily["Holiday"] = hol.resample("D").first()

elec = daily.reset_index()
elec["Weekday"] = ((elec["ds"].dt.dayofweek < 5)
                   & ~elec["Holiday"]).astype(int)
elec["Temp_sq"] = elec["Temperature"] ** 2
elec["unique_id"] = "VIC"
elec = elec[["unique_id", "ds", "Demand",
             "Temperature", "Temp_sq", "Weekday"]]

sf = StatsForecast(models=[AutoARIMA(season_length=7)],
                   freq="D")
sf = sf.fit(df=elec, target_col="Demand")
model = sf.fitted_[0, 0].model_
print(ARIMASummary(model))
# Regression with ARIMA(2,1,2)(2,0,2)[7] errors
print(pd.Series(model["coef"]).round(3).tail(3))
# ex_1    -7.501   (Temperature)
# ex_2     0.178   (Temp_sq)
# ex_3    29.325   (Weekday)
print(acorr_ljungbox(model["residuals"], lags=[14], model_df=8))
#     lb_stat  lb_pvalue
# 14   21.871    0.00128

from utilsforecast.processing import make_future_dataframe
future_elec = make_future_dataframe(sf.uids, sf.last_dates,
                                    freq="D", h=14)
future_elec["Temperature"] = 26.0
future_elec["Temp_sq"] = 26.0 ** 2
future_elec["Weekday"] = (future_elec["ds"].dt.dayofweek
                          < 5).astype(int)
hol = future_elec["ds"] == "2015-01-01"  # public holiday
future_elec.loc[hol, "Weekday"] = 0
fc_elec = sf.predict(h=14, X_df=future_elec, level=[80, 95])
fc_elec.round(1)

## [Slide 39-40] 10.4 Trend Models: Fitting in Python

In [ ]:
import pandas as pd
import numpy as np
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA
from statsforecast.arima import ARIMASummary

ap = pd.read_csv("data/aus_airpassengers.csv")
aus_pass = pd.DataFrame({
    "unique_id": "AUS",
    "ds": pd.to_datetime(ap["Year"], format="%Y"),
    "y": ap["Passengers"],
})
# deterministic trend: intercept and t = 1, ..., T
aus_pass["intercept"] = 1.0
aus_pass["trend"] = np.arange(1, len(aus_pass) + 1)

# Deterministic: d=0, intercept + linear trend as regressors
sf_det = StatsForecast(models=[AutoARIMA(d=0, max_p=1)],
                       freq="YS")
sf_det = sf_det.fit(df=aus_pass, target_col="y")
# Regression with ARIMA(1,0,0) errors:
#   ar1 0.956, intercept 0.902, trend 1.415

# Stochastic: d=1, no regressors (drift chosen automatically)
sf_sto = StatsForecast(models=[AutoARIMA(d=1)], freq="YS")
sf_sto = sf_sto.fit(df=aus_pass[["unique_id", "ds", "y"]])
# ARIMA(0,1,0) with drift: drift 1.419
for nm, m in (("deterministic", sf_det), ("stochastic", sf_sto)):
    mm = m.fitted_[0, 0].model_
    print(nm, ARIMASummary(mm), pd.Series(mm["coef"]).round(3).to_dict(),
          "sigma2 %.3f AICc %.2f" % (mm["sigma2"], mm["aicc"]))

## [Slide 42] 10.4 Forecasting: Deterministic vs Stochastic Trend

In [ ]:
from utilsforecast.processing import make_future_dataframe
h = 10
future = make_future_dataframe(uids=sf_det.uids,
                               last_times=sf_det.last_dates,
                               freq="YS", h=h)
future["intercept"] = 1.0
future["trend"] = np.arange(len(aus_pass) + 1,
                            len(aus_pass) + h + 1)
fc_det = sf_det.predict(h=h, X_df=future, level=[80, 95])
fc_sto = sf_sto.predict(h=h, level=[80, 95])
print(fc_det.round(1)); print(fc_sto.round(1))
# h=10: det 84.9 (95% width 21.5), sto 86.8 (95% width 25.6)

## [Slide 47] 10.5 Example: Café/Restaurant Expenditure

In [ ]:
import numpy as np
import pandas as pd

industry = "Cafes, restaurants and takeaway food services"
aus_cafe = (
    pd.read_csv(BASE + "aus_retail.csv", parse_dates=["Month"])
    .loc[lambda x: x["Month"].between("2004", "2018-12-01")
         & (x["Industry"] == industry)]
    .rename(columns={"Month": "ds", "Turnover": "y"})
    .groupby("ds", as_index=False).agg({"y": "sum"})
    .assign(unique_id="aus_cafe")
)
aus_cafe.set_index("ds")["y"].plot(     # time plot
    figsize=(8, 3), xlabel="", ylabel="$ (millions)")

## [Slide 49] 10.5 Example: Café/Restaurant Expenditure (cont.)

In [ ]:
from functools import partial
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA
from statsforecast.arima import ARIMASummary
from utilsforecast.feature_engineering import fourier, pipeline

aus_cafe["y"] = np.log(aus_cafe["y"])   # log transform
results = {}
for k in range(1, 7):                  # K = 1, ..., 6
    features = [partial(fourier, season_length=12, k=k)]
    train, fut = pipeline(aus_cafe, features=features,
                          freq="MS", h=24)
    sf = StatsForecast(
        models=[AutoARIMA(season_length=12, seasonal=False)],
        freq="MS")
    sf = sf.fit(df=train)
    model = sf.fitted_[0, 0].model_
    fc = sf.predict(h=24, X_df=fut, level=[80, 95])
    num = fc.select_dtypes(np.number).columns
    fc[num] = np.exp(fc[num])          # back-transform
    results[k] = (model["aicc"], fc)
    print(f"K={k}: {ARIMASummary(model)}  "
          f"AICc = {model['aicc']:.2f}")
# K=5: ARIMA(0,1,1) errors, AICc = -917.48 (minimum); K=6: -915.19

## [Slide 67-68] 10.6 Insurance Example: Code

In [ ]:
import pandas as pd
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA

insurance = pd.read_csv("data/insurance.csv",
                        parse_dates=["ds"])
# lag-1 predictor (chosen by AICc); first row dropped
fit_df = insurance.assign(
    TVadverts_1=insurance["TVadverts"].shift(1)).dropna()

sf = StatsForecast(models=[AutoARIMA(d=0)], freq="MS")
sf = sf.fit(df=fit_df[["unique_id", "ds", "Quotes",
                       "TVadverts", "TVadverts_1"]],
            target_col="Quotes")
model = sf.fitted_[0, 0].model_
print(pd.Series(model["coef"]).round(3))
# ar1 0.512, ma1 0.917, ma2 0.459,
# intercept 2.155, ex_1 1.253, ex_2 0.146

# Forecast 12 months with advertising = 8 units
future_adv = pd.DataFrame({
    "unique_id": "insurance",
    "ds": pd.date_range("2005-05-01", periods=12, freq="MS"),
    "TVadverts": 8.0,
    "TVadverts_1": [insurance["TVadverts"].iloc[-1]]
                   + [8.0]*11,
})
fc = sf.predict(h=12, X_df=future_adv, level=[80, 95])
fc.round(2)